# Vídeo-aulas com voz clonada no cluster vlab (UFSC)

Instala e roda `gerador_de_video_e_som.py` (Chatterbox Multilingual + whisperx, ou F5-TTS pt-BR) num nó com GPU do vlab.

**Ordem de uso**
1. Seções 1–3 uma única vez (ambiente, modelos, arquivos) — **num nó com acesso à internet** (normalmente o nó de login ou um nó interativo).
2. Seção 4 a cada aula — **num nó com GPU** (via Slurm ou sessão interativa).

**Antes de começar, envie para a sua `$HOME` no cluster** (scp, rsync, ou o gerenciador de arquivos do JupyterHub):
```
~/mesoescala-montanhas/video/gerador_de_video_e_som.py
~/mesoescala-montanhas/video/voz_referencia/amostra_nova.mp3
~/mesoescala-montanhas/Capitulo1-introducao.pptx   (e demais .pptx)
```
Adapte `PROJ` na próxima célula se usar outra pasta.

In [ ]:
import os, sys, subprocess, pathlib, shutil

PROJ    = pathlib.Path.home() / "mesoescala-montanhas"     # pasta do projeto no cluster
VIDEO   = PROJ / "video"
ENV     = "voz"                                             # nome do ambiente conda/mamba
HF_HOME = pathlib.Path.home() / ".cache" / "huggingface"    # cache de modelos (fora de /tmp e de scratch efêmero!)

os.environ["HF_HOME"] = str(HF_HOME)
print("Projeto:", PROJ, "| existe:", PROJ.exists())
print("Script :", VIDEO / "gerador_de_video_e_som.py", "| existe:", (VIDEO / "gerador_de_video_e_som.py").exists())
print("Amostra:", VIDEO / "voz_referencia" / "amostra_nova.mp3", "| existe:", (VIDEO / "voz_referencia" / "amostra_nova.mp3").exists())
print("GPU    :", shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "nvidia-smi não encontrado (nó sem GPU ou módulo CUDA não carregado)")
print("Módulos:", "module disponível" if shutil.which("module") or os.environ.get("MODULESHOME") else "sem environment-modules")

## 1. Ambiente (uma vez)

O vlab normalmente oferece `module load`. Veja o que existe para CUDA, conda e LibreOffice; se não houver conda nos módulos, a célula seguinte instala um Miniforge na sua `$HOME`.

In [ ]:
!bash -lc "module avail 2>&1 | grep -iE 'cuda|conda|mamba|miniforge|anaconda|libreoffice|ffmpeg' || echo 'nenhum módulo relevante listado'"

In [ ]:
# Instala Miniforge na $HOME se não houver conda/mamba disponível
mf = pathlib.Path.home() / "miniforge3"
if shutil.which("mamba") or shutil.which("conda") or (mf / "bin" / "mamba").exists():
    print("conda/mamba já disponível em", shutil.which("mamba") or shutil.which("conda") or mf)
else:
    !wget -q -O /tmp/miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /tmp/miniforge.sh -b -p $HOME/miniforge3
    print("Miniforge instalado em", mf)

In [ ]:
%%bash -l
# Cria o ambiente "voz" com Python 3.11 + ffmpeg + LibreOffice + poppler (conversão dos slides) + fontes
export PATH=$HOME/miniforge3/bin:$PATH
MAMBA=$(command -v mamba || command -v conda)
if $MAMBA env list | grep -q "^voz "; then echo "ambiente voz já existe"; else
  $MAMBA create -y -n voz -c conda-forge python=3.11 ffmpeg libreoffice poppler fonts-conda-ecosystem font-ttf-dejavu-sans-mono
fi
$MAMBA env list

In [ ]:
%%bash -l
export PATH=$HOME/miniforge3/bin:$PATH
source $(dirname $(command -v mamba || command -v conda))/../etc/profile.d/conda.sh
conda activate voz
# torch com CUDA (cu126 serve para drivers >= 560; use cu121 se o driver do cluster for mais antigo — veja nvidia-smi)
pip install -q torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
pip install -q chatterbox-tts whisperx f5-tts faster-whisper python-pptx edge-tts ipykernel
# garante que o chatterbox não trocou o torch por uma build CPU
python -c "import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())"
python -m ipykernel install --user --name voz --display-name "Python (voz)"

> Após esta célula, troque o kernel deste notebook para **Python (voz)** (menu *Kernel → Change kernel*) e reexecute a célula de configuração do topo. Se `CUDA: False` apareceu num nó com GPU, o `pip install chatterbox-tts` substituiu o torch: repita só a linha do `pip install torch==...` com `--force-reinstall --no-deps`.

## 2. Modelos (uma vez, com internet)

Baixa Chatterbox (~3 GB), wav2vec2-pt para alinhamento (~1,2 GB) e o F5-TTS pt-BR (~1,3 GB) para `HF_HOME`. Depois disso o script roda 100 % offline — importante porque nós de cálculo geralmente não têm saída para a internet.

In [ ]:
os.environ.pop("HF_HUB_OFFLINE", None)
from huggingface_hub import snapshot_download
for repo in ["ResembleAI/chatterbox", "jonatasgrosman/wav2vec2-large-xlsr-53-portuguese", "traderpedroso/F5-TTS-BRAZILIAN-PORTUGUESE"]:
    print("->", repo); snapshot_download(repo)
# vocoder do F5 e modelo whisper 'small' (transcrição da referência) também ficam em cache
snapshot_download("charactr/vocos-mel-24khz")
from faster_whisper import WhisperModel; WhisperModel("small", device="cpu", compute_type="int8")
print("Cache em", HF_HOME, "— pronto para modo offline.")

In [ ]:
# Verificação offline: deve carregar sem tocar na rede
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
m = ChatterboxMultilingualTTS.from_pretrained(device="cuda" if torch.cuda.is_available() else "cpu")
print("Chatterbox OK | VRAM alocada:", round(torch.cuda.memory_allocated()/1e9, 2), "GB" if torch.cuda.is_available() else "")
del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 2b. Escolher a GPU (nó compartilhado)

Os nós do vlab têm várias H100 **divididas com outros usuários**. Esta célula encerra execuções suas que tenham ficado para trás e escolhe a placa com memória livre e utilização baixa em duas leituras seguidas. Rode-a **antes de cada** síntese (seções 3 e 4).

In [ ]:
import subprocess, time, os, signal

# 1) Encerra instâncias anteriores do gerador (evita disputar a GPU consigo mesmo)
meu = os.getpid()
ps = subprocess.run(["pgrep", "-f", "gerador_de_video_e_som.py"], capture_output=True, text=True).stdout.split()
for pid in ps:
    if int(pid) != meu:
        try: os.kill(int(pid), signal.SIGKILL); print("encerrado processo antigo", pid)
        except ProcessLookupError: pass

# 2) Duas leituras das GPUs, 5 s de intervalo; exige >= MIN_LIVRE MiB livres e utilizacao <= MAX_UTIL em ambas
MIN_LIVRE, MAX_UTIL = 6000, 10
def ler():
    out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    return {int(i): (int(f), int(u)) for i, f, u in (l.split(",") for l in out.strip().splitlines())}
a = ler(); time.sleep(5); b = ler()
print(f"{'GPU':>3} {'livre MiB':>10} {'util%':>6}   (2ª leitura)")
cands = []
for i in sorted(a):
    f = min(a[i][0], b[i][0]); u = max(a[i][1], b[i][1])
    ok = f >= MIN_LIVRE and u <= MAX_UTIL
    print(f"{i:>3} {f:>10} {u:>6}   {'<- candidata' if ok else ''}")
    if ok: cands.append((u, -f, i))
if not cands:
    raise SystemExit("Nenhuma GPU com memória livre e ociosa agora. Espere alguns minutos e rode esta célula de novo, "
                     "ou use um job Slurm com GPU reservada (seção 4, opção B).")
GPU = sorted(cands)[0][2]
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
print(f"\nUsando GPU {GPU}  (CUDA_VISIBLE_DEVICES={GPU})")

## 3. Teste da voz (um slide)

Rode no nó com GPU. A flag `--online` **não** é usada: tudo vem do cache.

In [ ]:
PPTX = PROJ / "Capitulo1-introducao.pptx"     # <- aula a renderizar
VOZ  = "reinaldo_haas"                          # "reinaldo_haas" (Chatterbox) ou "reinaldo_f5" (F5-TTS pt-BR, mais leve)

cmd = [sys.executable, str(VIDEO / "gerador_de_video_e_som.py"), str(PPTX), "-s", "1", "-v", VOZ]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=PROJ, text=True, env=os.environ)
print("exit:", r.returncode)

In [ ]:
# Ouça o resultado aqui mesmo
from IPython.display import Audio, display
wav = sorted((PROJ / "video_work").rglob("slide_001.wav"))
display(Audio(str(wav[-1]))) if wav else print("slide_001.wav não encontrado — veja a saída acima")

Se a voz não convencer, teste `--cfg 0.3` (mais fiel ao timbre) ou `-v reinaldo_f5`; o cache regenera sozinho quando os parâmetros mudam.

## 4. Aula completa

**Opção A — sessão interativa com GPU** (JupyterHub já num nó de GPU): execute a célula abaixo.
**Opção B — job Slurm** (recomendado para aulas inteiras; sobrevive ao fechamento do navegador): gere e envie o script da célula seguinte.

In [ ]:
# Opção A: direto nesta sessão (precisa estar num nó com GPU). PARALELO = nº de trabalhadores na mesma GPU
PARALELO = 8          # H100 (80 GB): 8; GPU de 24 GB: 4-5; cada trabalhador usa ~3,5 GB de VRAM
cmd = [sys.executable, str(VIDEO / "gerador_de_video_e_som.py"), str(PPTX), "-v", VOZ, "--paralelo", str(PARALELO)]
r = subprocess.run(cmd, cwd=PROJ, text=True, env=os.environ); print("exit:", r.returncode)

In [ ]:
# Opção B: job Slurm. Ajuste --partition / --gres / --time conforme o vlab (veja `sinfo` e as instruções do cluster).
job = f"""#!/bin/bash
#SBATCH --job-name=videoaula
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=03:00:00
#SBATCH --output={PROJ}/video_work/slurm-%j.out

export PATH=$HOME/miniforge3/bin:$PATH
source $(dirname $(command -v mamba || command -v conda))/../etc/profile.d/conda.sh
conda activate {ENV}
export HF_HOME={HF_HOME}
cd {PROJ}
# Em nó compartilhado sem --exclusive: escolhe a GPU mais livre
export CUDA_VISIBLE_DEVICES=$(nvidia-smi --query-gpu=index,memory.free,utilization.gpu --format=csv,noheader,nounits | sort -t, -k3,3n -k2,2nr | head -1 | cut -d, -f1)
echo "GPU escolhida: $CUDA_VISIBLE_DEVICES"
python {VIDEO}/gerador_de_video_e_som.py {PPTX.name} -v {VOZ} --paralelo 8 "$@"
"""
(PROJ / "video_work").mkdir(exist_ok=True)
sb = PROJ / "video_work" / "videoaula.sbatch"; sb.write_text(job); print(sb.read_text())
print(subprocess.run(["sbatch", str(sb)], capture_output=True, text=True, cwd=PROJ))

In [ ]:
!squeue -u $USER
!tail -n 30 $HOME/mesoescala-montanhas/video_work/slurm-*.out 2>/dev/null | tail -n 30

## 5. Resultado

O `.mp4` final fica em `video/` dentro do projeto. Baixe com scp/rsync ou pelo gerenciador de arquivos do JupyterHub:
```
rsync -avP vlab:~/mesoescala-montanhas/video/*.mp4 ./
```

**Problemas comuns**
- `GPU nao detectada`: a sessão/job não está num nó de GPU, ou `module load cuda` não foi feito antes de abrir o Jupyter.
- `um modelo não está no cache local`: a seção 2 não rodou, ou `HF_HOME` aponta para outro lugar (confira `echo $HF_HOME` no job).
- `LibreOffice (soffice) não encontrado`: ative o ambiente `voz` no job (o conda-forge instala o `soffice`); alternativa `module load libreoffice`.
- Legendas sem acento ou fonte estranha: `fc-list | grep -i dejavu` deve listar DejaVu Sans; se não, `mamba install -c conda-forge font-ttf-dejavu-sans-mono fontconfig`.